# Carbon Taxes and CO2 Emissions: A Synthetic-Control Analysis in Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/starter-academic-v501/blob/master/content/post/python_sc_co2tax/notebook.ipynb)

Runnable companion to the blog post
[*Carbon Taxes and CO2 Emissions*](https://carlos-mendez.org/post/python_sc_co2tax/).

**Inspired by** the [RTutor problem set](https://github.com/TheresaGraefe/RTutorCarbonTaxesAndCO2Emissions)
by Theresa Graefe (2020), which replicates
[Andersson (2019), *AEJ: Economic Policy* 11(4)](https://doi.org/10.1257/pol.20170144).
All empirical findings are Andersson's.

## The question we are answering

In **1991**, Sweden put a price on carbon dioxide. The reform combined a new **carbon tax**, a new **VAT** on transport fuel (1990), and a partial cut to a pre-existing **energy tax**. Thirty years later we ask:

1. **Did the carbon tax actually reduce CO2 emissions from transport?**
2. **Did it cost Sweden any economic growth?**

## How this notebook is organised

Each analytical step is its own short code cell. Read the markdown just *above* a code cell to learn what is about to happen. Read the markdown just *below* to interpret what came out. Run cells in order — many depend on variables defined earlier.

## 1. Install dependencies (Colab)

Run once per session. We use four specialised packages on top of `pandas`, `numpy`, and `matplotlib`:

- [`pysyncon`](https://sdfordham.github.io/pysyncon/) — synthetic control
- [`pyfixest`](https://pyfixest.org/) — OLS and IV regressions
- [`statsmodels`](https://www.statsmodels.org/) — Newey–West HAC standard errors
- [`pyreadr`](https://github.com/ofajardo/pyreadr) — read R `.Rds` files in Python

In [ ]:
%pip install -q pyfixest pysyncon pyreadr statsmodels pandas numpy matplotlib

## 2. Imports and dark theme

We import everything once and set the matplotlib theme globally. Every figure inherits the dark navy look.

In [ ]:
from __future__ import annotations
from pathlib import Path
import io, urllib.request, tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadr
import statsmodels.api as sm
import pyfixest as pf
from pysyncon import Dataprep, Synth

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Site color palette ──────────────────────────────────────────────────────
STEEL_BLUE  = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK  = "#141413"
TEAL        = "#00d4c8"

DARK_NAVY  = "#0f1729"
GRID_LINE  = "#1f2b5e"
LIGHT_TEXT = "#c8d0e0"
WHITE_TEXT = "#e8ecf2"

plt.rcParams.update({
    "figure.facecolor": DARK_NAVY, "axes.facecolor": DARK_NAVY,
    "axes.edgecolor": DARK_NAVY, "axes.linewidth": 0,
    "axes.labelcolor": LIGHT_TEXT, "axes.titlecolor": WHITE_TEXT,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.spines.bottom": False,
    "axes.grid": True, "grid.color": GRID_LINE,
    "grid.linewidth": 0.6, "grid.alpha": 0.8,
    "xtick.color": LIGHT_TEXT, "ytick.color": LIGHT_TEXT,
    "xtick.major.size": 0, "ytick.major.size": 0,
    "text.color": WHITE_TEXT, "font.size": 12,
    "legend.frameon": False, "legend.fontsize": 11,
    "legend.labelcolor": LIGHT_TEXT,
    "figure.edgecolor": DARK_NAVY,
    "savefig.facecolor": DARK_NAVY, "savefig.edgecolor": DARK_NAVY,
})

SLUG = "python_sc_co2tax"
ROOT = Path.cwd()  # Notebook-friendly; works on Colab and locally.

SAVE_KW = dict(dpi=150, bbox_inches="tight",
               facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0.05)

def savefig(name: str) -> None:
    """Save and show the current figure."""
    plt.savefig(ROOT / f"{SLUG}_{name}.png", **SAVE_KW)
    plt.show()
    plt.close()

def hline(title: str) -> None:
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

print("Setup complete. Figures will be saved to:", ROOT)


## 3. Data sources (direct GitHub URLs)

We load **six datasets** straight from their GitHub raw URLs. No local files needed.

| Dataset | Used for |
| --- | --- |
| `carbontax_data.dta` | DiD + Synthetic Sweden (OECD panel) |
| `descr_Sweden.Rds` | Descriptive plots + GDP-gap analysis |
| `GDP_data.Rds` | Synthetic-GDP exercise |
| `regression_data.Rds` | OLS + IV regressions |
| `leave_one_out_data.dta` | Robustness plot |
| `disentangling_data.dta` | Carbon-tax-vs-VAT decomposition |

In [ ]:
BASE_URL = (
    "https://raw.githubusercontent.com/cmg777/starter-academic-v501/master/"
    "content/post/python_sc_co2tax/references/RTutorCarbonTax-master/"
    "inst/ps/CarbonTaxesAndCO2Emissions/material"
)
URL = {fname: f"{BASE_URL}/{fname}" for fname in [
    "carbontax_data.dta",
    "disentangling_data.dta",
    "leave_one_out_data.dta",
    "descr_Sweden.Rds",
    "regression_data.Rds",
    "GDP_data.Rds",
]}

def read_rds(url: str) -> pd.DataFrame:
    """Stream an .Rds file from a URL and return its sole data frame."""
    with urllib.request.urlopen(url) as resp, \
         tempfile.NamedTemporaryFile(suffix=".Rds", delete=False) as tmp:
        tmp.write(resp.read())
        tmp_path = tmp.name
    df = pyreadr.read_r(tmp_path)[None].reset_index(drop=True)
    return df

for k, u in URL.items():
    print(f"  {k:<28s}  {u}")


## 4. Load the six datasets

A **panel dataset** has multiple units (countries) observed across multiple time periods (years). Our outcome of interest is per-capita CO2 emissions from transport in metric tons.

Three numbers about the time window matter:

- **46 years total** (1960–2005).
- **30 pre-treatment years** (1960–1989) to fit the counterfactual.
- **16 post-treatment years** (1990–2005) to measure the effect.

In [ ]:
panel  = pd.read_stata(URL["carbontax_data.dta"])
loo    = pd.read_stata(URL["leave_one_out_data.dta"])
disent = pd.read_stata(URL["disentangling_data.dta"])

descr_sweden = read_rds(URL["descr_Sweden.Rds"])
gdp_data     = read_rds(URL["GDP_data.Rds"])
reg_data     = read_rds(URL["regression_data.Rds"])

print(f"panel (carbontax_data.dta): {panel.shape}, countries={panel['country'].nunique()}, years={panel['year'].min():.0f}-{panel['year'].max():.0f}")
print(f"descr_Sweden.Rds:          {descr_sweden.shape}")
print(f"GDP_data.Rds:              {gdp_data.shape}, countries={gdp_data['country'].nunique()}")
print(f"regression_data.Rds:       {reg_data.shape}, years={reg_data['year'].min():.0f}-{reg_data['year'].max():.0f}")
print(f"disentangling_data.dta:    {disent.shape}")
print(f"leave_one_out_data.dta:    {loo.shape}")

# 5. Descriptive overview

A good causal study starts with descriptive plots. We *look* at the policy variable (taxes), the outcome (CO2), and the obvious mechanism (fuel consumption) before running any model.

## 5.1 Decomposing Sweden's gasoline price

Sweden's retail gasoline price has four parts:

1. **Wholesale price** — set by world oil markets.
2. **Energy tax** — long-standing, predates the reform.
3. **Carbon tax** — new in 1991.
4. **VAT** — new on fuel in 1990.

First we peek at the raw price series for Sweden.

In [ ]:
ds = descr_sweden.copy()
print(ds[["year", "VAT", "en_tax", "CO2_tax", "pw_real", "total_tax"]].head().to_string(index=False))

Now the layered plot of all four price components.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(ds["year"], ds["pw_real"], color=STEEL_BLUE, lw=2.2, label="Real wholesale price")
ax.plot(ds["year"], ds["en_tax"], color=WARM_ORANGE, lw=2.0, label="Energy tax")
ax.plot(ds["year"], ds["CO2_tax"], color=TEAL, lw=2.0, label="Carbon tax")
ax.plot(ds["year"], ds["VAT"], color="#c179c8", lw=1.8, label="VAT")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Real price components (SEK / litre)")
ax.set_title("Sweden — gasoline price decomposition (1960–2005)")
ax.legend(loc="upper left", ncol=2)
savefig("gasoline_price_components")

**Notice:**

- The carbon tax (teal) appears in 1991 and grows steadily.
- The energy tax (orange) actually *drops* — the reform was partly a **tax swap**, not a pure tax hike.
- The wholesale price (blue) reflects 1970s–80s oil shocks, not Swedish policy.

Now we sum the parts into the actual retail price consumers face.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
retail = ds["pw_real"] + ds["total_tax"]
ax.plot(ds["year"], retail, color=TEAL, lw=2.4, label="Retail gasoline price (real)")
ax.plot(ds["year"], ds["total_tax"], color=WARM_ORANGE, lw=2.0, label="Total tax")
ax.plot(ds["year"], ds["pw_real"], color=STEEL_BLUE, lw=2.0, label="Real wholesale price")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("SEK / litre (real)")
ax.set_title("Sweden — retail price = wholesale + total tax")
ax.legend(loc="upper left")
savefig("retail_price")

After 1990 the retail price rises mostly because taxes rise — the wholesale price actually falls. By 2005 the carbon tax matches the energy tax in size.

## 5.2 CO2 emissions and fuel consumption

Did the reform actually cut emissions? Two plots side by side:

- Left: Sweden CO2 vs OECD mean.
- Right: Sweden gasoline vs diesel consumption.

The right plot matters because a CO2 reduction could come from **less driving** *or* from **fuel substitution** (gasoline → diesel).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(ds["year"], ds["CO2_Sweden"], color=WARM_ORANGE, lw=2.2)
axes[0].plot(ds["year"], ds["CO2_OECD"], color=STEEL_BLUE, lw=2.0, ls="--", label="OECD sample mean")
axes[0].axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
axes[0].set_title("Per-capita CO$_2$ from transport")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Metric tons / capita")
axes[0].legend(["Sweden", "OECD sample mean"], loc="lower right")

axes[1].plot(ds["year"], ds["gas_cons"], color=TEAL, lw=2.2, label="Gasoline")
axes[1].plot(ds["year"], ds["diesel_cons"], color=WARM_ORANGE, lw=2.0, label="Diesel")
axes[1].axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
axes[1].set_title("Per-capita fuel consumption — Sweden")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("kg oil-equivalent / capita")
axes[1].legend(loc="lower right")

plt.suptitle("Outcomes around the 1990 Swedish energy-tax reform",
             color=WHITE_TEXT, fontsize=13)
savefig("co2_vs_consumption")

Before 1990 Sweden's CO2 tracks the OECD mean. After 1990 Sweden plateaus while the OECD keeps climbing — the first visible sign of an effect. Diesel rises throughout, gasoline falls — so part of the reduction is **fuel substitution**.

Finally, let's see where Sweden sits in the donor distribution. Each panel below is one country's CO2 path.

In [ ]:
countries = sorted(panel["country"].unique())
fig, axes = plt.subplots(3, 5, figsize=(15, 8.5), sharex=True, sharey=True)
for ax, country in zip(axes.ravel(), countries):
    sub = panel[panel["country"] == country].sort_values("year")
    color = WARM_ORANGE if country == "Sweden" else STEEL_BLUE
    lw = 2.4 if country == "Sweden" else 1.4
    ax.plot(sub["year"], sub["CO2_transport_capita"], color=color, lw=lw)
    ax.axvline(1990, color=LIGHT_TEXT, lw=0.6, ls=":")
    ax.set_title(country, color=WHITE_TEXT, fontsize=11)
plt.suptitle("CO$_2$ from transport, per capita — OECD donor pool",
             color=WHITE_TEXT, fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
savefig("co2_donor_pool")

Sweden (orange) sits in the middle of the donor distribution before 1990, neither highest (US, Canada) nor lowest (Portugal, Poland). Many donors are plausible matches.

# 6. Estimating causal effects

We now try three estimators, from worst to best.

## 6.1 The naive Sweden-only before/after

The simplest model regresses Sweden's CO2 on a 0/1 post-1990 dummy:

$$\text{CO2}_{\text{Sweden},t} = \alpha + \delta \cdot \mathbf{1}\{t \geq 1990\} + \varepsilon_t$$

It treats every other thing that changed in Sweden between 1960 and 2005 (population, income, vehicle stock) as if it were part of the policy effect. **This will give the wrong sign.**

First, prepare the indicator variables on the panel.

In [ ]:
panel = panel.copy()
panel["post"] = (panel["year"] >= 1990).astype(int)
panel["treated"] = (panel["country"] == "Sweden").astype(int)
panel["Sweden_post"] = panel["treated"] * panel["post"]

Now the naive Sweden-only regression.

In [ ]:
sw = panel[panel["country"] == "Sweden"].copy()
sw["delta"] = (sw["year"] >= 1990).astype(int)
m_time = pf.feols("CO2_transport_capita ~ delta", data=sw, vcov="HC1")
print("Sweden time-difference regression:")
print(m_time.tidy().round(4))

**Naive estimate:** +0.55 t/capita (t = 7.0). Sweden *looks* dirtier after 1990 — because the window includes 16 years of growth. Wrong answer.

## 6.2 Difference-in-differences (DiD)

DiD compares Sweden's pre-vs-post change to a *control*'s pre-vs-post change. The leftover is the "extra" change in Sweden — hopefully the treatment.

$$y_{jt} = \beta_0 + \beta_1 T_j + \beta_2 P_t + \beta_3 (T_j \cdot P_t) + \varepsilon_{jt}$$

$T_j = 1$ if Sweden, $P_t = 1$ if post-1990. **$\beta_3$ is the DiD coefficient** — the average treatment effect on the treated (ATT) under the **parallel-trends** assumption.

### Sweden vs Denmark only

In [ ]:
two = panel[panel["country"].isin(["Sweden", "Denmark"])].copy()
m_did2 = pf.feols(
    "CO2_transport_capita ~ treated + post + Sweden_post",
    data=two, vcov="HC1",
)
print("DiD: Sweden vs Denmark (HC1):")
print(m_did2.tidy().round(4))

**Sweden vs Denmark:** −0.140 t/capita. Sign flipped! But underpowered with only one control (p = 0.23).

### Sweden vs full OECD donor pool

Expand the control to all 14 other OECD countries. Use **cluster-robust standard errors** (one cluster per country) to handle serial correlation within each country's series.

In [ ]:
m_did_oecd = pf.feols(
    "CO2_transport_capita ~ treated + post + Sweden_post",
    data=panel, vcov={"CRV1": "country"},
)
print("DiD: Sweden vs full OECD donor pool (cluster by country):")
print(m_did_oecd.tidy().round(4))

**Sweden vs OECD:** −0.214 t/capita (p = 0.02). Statistically significant. Both DiD estimates are between 7% and 11% of Sweden's pre-reform level.

Save a comparison table and plot Sweden vs Denmark for visual inspection.

In [ ]:
did_tab = pd.concat(
    [m_did2.tidy().assign(model="Sweden vs Denmark"),
     m_did_oecd.tidy().assign(model="Sweden vs OECD")],
    axis=0
)
did_tab.to_csv(ROOT / "tab_did_comparison.csv")

fig, ax = plt.subplots(figsize=(9, 5.4))
for cty, col in [("Sweden", WARM_ORANGE), ("Denmark", STEEL_BLUE)]:
    sub = panel[panel["country"] == cty].sort_values("year")
    ax.plot(sub["year"], sub["CO2_transport_capita"], color=col, lw=2.2, label=cty)
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("DiD baseline — Sweden vs Denmark")
ax.legend(loc="lower right")
savefig("did_sweden_denmark")

**Parallel-trends check:** look at 1960–1990. Sweden and Denmark do not move in perfect lockstep — Sweden trends up more steeply. The parallel-trends assumption is questionable, motivating **synthetic control**.

# 7. Synthetic Sweden

DiD picks a single control country. Rigid. **Synthetic control** picks a *weighted blend* of donor countries (Denmark 30% + Belgium 27% + ...). The blend is chosen so the synthetic version matches real Sweden as closely as possible *before* the reform.

After the reform, the gap between Sweden and Synthetic Sweden is the estimated treatment effect.

### The math, briefly

Let $X_1$ be Sweden's pre-treatment predictors (GDP per capita, vehicles, gasoline consumption, urbanisation, plus three lagged CO2 levels). $X_0$ is the same predictors for each donor. We pick donor weights $w$ to solve:

$$w^* = \arg\min_{w} (X_1 - X_0 w)^\top V (X_1 - X_0 w) \quad \text{s.t.} \quad w_j \geq 0, \; \sum_j w_j = 1$$

`pysyncon` does the nested optimisation behind two objects: `Dataprep` packs the matrices, `Synth.fit()` runs the optimiser.

## 7.1 Pack the data — `Dataprep`

We specify:

- **`predictors`** — four economic covariates averaged over 1980–1989.
- **`special_predictors`** — three single-year CO2 levels (1989, 1980, 1970) to anchor the fit on the outcome itself.
- **`treatment_identifier`** — Sweden.
- **`controls_identifier`** — the other 14 countries.
- **`time_optimize_ssr`** — the pre-treatment window the optimiser sees.

In [ ]:
controls = [c for c in countries if c != "Sweden"]
dataprep = Dataprep(
    foo=panel,
    predictors=["GDP_per_capita", "vehicles_capita", "gas_cons_capita", "urban_pop"],
    predictors_op="mean",
    time_predictors_prior=range(1980, 1990),
    special_predictors=[
        ("CO2_transport_capita", [1989], "mean"),
        ("CO2_transport_capita", [1980], "mean"),
        ("CO2_transport_capita", [1970], "mean"),
    ],
    dependent="CO2_transport_capita",
    unit_variable="country",
    time_variable="year",
    treatment_identifier="Sweden",
    controls_identifier=controls,
    time_optimize_ssr=range(1960, 1990),
)
print("Dataprep ready:", len(controls), "controls,",
      len(dataprep.predictors), "regular predictors,",
      len(dataprep.special_predictors), "special predictors")

## 7.2 Fit the optimiser — `Synth.fit()`

This runs the nested loop: pick predictor importance weights $V$, then pick donor weights $w$ that minimise the pre-period gap. `Nelder–Mead` is the default for the outer loop.

In [ ]:
synth = Synth()
synth.fit(dataprep=dataprep, optim_method="Nelder-Mead", optim_initial="equal")

print("Donor weights for Synthetic Sweden (top 8):")
w_sorted = synth.weights().sort_values(ascending=False)
print(w_sorted.head(8).round(4))
print(f"\nWeights sum to {w_sorted.sum():.6f}")
v_labels = list(dataprep.predictors) + [f"{name}({yrs[0] if len(yrs)==1 else f'{yrs[0]}-{yrs[-1]}'})"
                                         for (name, yrs, _) in dataprep.special_predictors]
v_diag = np.diag(synth.V) if synth.V.ndim == 2 else np.asarray(synth.V).ravel()
print(f"\nV (predictor) weights:\n{pd.Series(v_diag, index=v_labels).round(4)}")
print(f"\nLoss V (MSPE pre): {synth.loss_V:.6f}")
print(f"Loss W:           {synth.loss_W:.6f}")

**Six donors carry all the weight:** Denmark, Belgium, New Zealand, Greece, USA, Switzerland. The other 9 get essentially zero. Denmark and Belgium together hold more than half — the most demographically and economically similar peers.

## 7.3 Build Sweden's actual vs synthetic series

`Synth.fit` gave us the weights. To get Synthetic Sweden's CO2 path year-by-year, we multiply each donor's actual CO2 by its weight and sum across donors. The treatment gap is `Sweden − Synthetic Sweden`.

In [ ]:
years = np.arange(1960, 2006)
y_sweden = panel[panel["country"] == "Sweden"].set_index("year").loc[years, "CO2_transport_capita"]
panel_wide = panel.pivot(index="year", columns="country", values="CO2_transport_capita")
y_synth = panel_wide.loc[years, controls] @ w_sorted.reindex(controls).fillna(0)

actual_vs_synth = pd.DataFrame({"sweden": y_sweden.values,
                                "synth_sweden": y_synth.values,
                                "gap": y_sweden.values - y_synth.values},
                               index=years)
actual_vs_synth.index.name = "year"
actual_vs_synth.to_csv(ROOT / "tab_synth_sweden.csv")

print(f"Sweden 2005:       {y_sweden.loc[2005]:.4f} t/capita")
print(f"Synth Sweden 2005: {y_synth.loc[2005]:.4f} t/capita")
print(f"Gap 2005:          {y_sweden.loc[2005] - y_synth.loc[2005]:.4f} t/capita "
      f"({(y_sweden.loc[2005]-y_synth.loc[2005])/y_synth.loc[2005]*100:.2f}% vs synth)")

post = (years >= 1990)
gap = y_sweden.values - y_synth.values
avg_post_pct = (gap[post] / y_synth.values[post]).mean() * 100
print(f"Average post-treatment gap (1990-2005): {gap[post].mean():.4f} t/capita ({avg_post_pct:.2f}%)")

**Headline:** average 11.3% reduction per year over 1990–2005. By 2005 the gap is −0.36 t/capita (−15% of synthetic). Roughly **one ton of avoided CO2 per capita every 3.7 years**.

## 7.4 Path plot — actual vs synthetic

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(years, y_sweden.values, color=WARM_ORANGE, lw=2.4, label="Sweden")
ax.plot(years, y_synth.values, color=STEEL_BLUE, lw=2.2, ls="--", label="Synthetic Sweden")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("Path plot — Sweden vs Synthetic Sweden")
ax.legend(loc="lower right")
savefig("synth_sweden_fit")

Before 1990 the two lines overlap almost perfectly. After 1990 they split — Sweden plateaus, Synthetic Sweden keeps climbing.

## 7.5 Donor weights bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.4))
nonzero = w_sorted[w_sorted > 1e-4]
ax.barh(nonzero.index[::-1], nonzero.values[::-1], color=TEAL)
ax.set_xlabel("Donor weight in Synthetic Sweden")
ax.set_title("Country weights $w^*$ (Synthetic Sweden)")
for i, (lbl, v) in enumerate(zip(nonzero.index[::-1], nonzero.values[::-1])):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", color=LIGHT_TEXT, fontsize=10)
savefig("synth_weights")
w_sorted.to_csv(ROOT / "tab_synth_weights.csv")

Concentrated weights on a handful of donors → the optimiser found a tight fit. Spread-out weights would have been a red flag.

## 7.6 Treatment gap year-by-year

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(years, gap, color=WARM_ORANGE, lw=2.4)
ax.fill_between(years, gap, 0, where=(gap < 0), color=WARM_ORANGE, alpha=0.18)
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.axhline(0, color=LIGHT_TEXT, lw=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Gap = Sweden − Synthetic Sweden (t CO$_2$ / cap)")
ax.set_title("Treatment gap in CO$_2$ from transport")
savefig("synth_gap")

Pre-1990 the gap hugs zero. Post-1990 it grows steadily negative — the post-treatment effect, sustained for 16 years.

# 8. Placebo tests — is the gap real?

The optimiser is *designed* to make Sweden look unique in the post-period. We need to check the gap is not an artefact of the method itself.

Three falsification tests:

| Test | What we do | What we expect if effect is real |
| --- | --- | --- |
| In-time | Backdate treatment to 1980 | No gap between 1980 and 1990 |
| In-space | Re-run SCM treating each donor as "treated" | Sweden's gap larger than placebo gaps |
| Leave-one-out | Drop each high-weight donor | Headline gap barely moves |

## 8.1 In-time placebo — pretend the reform happened in 1980

Fit a synthetic control using only pre-1980 data, then look at the 1980–1990 gap. If there was no reform in 1980, the gap should be flat. If it isn't, the SCM invents gaps and we should distrust the headline result.

In [ ]:
dp_time = Dataprep(
    foo=panel,
    predictors=["GDP_per_capita", "vehicles_capita", "gas_cons_capita", "urban_pop"],
    predictors_op="mean",
    time_predictors_prior=range(1970, 1980),
    special_predictors=[
        ("CO2_transport_capita", [1979], "mean"),
        ("CO2_transport_capita", [1970], "mean"),
        ("CO2_transport_capita", [1965], "mean"),
    ],
    dependent="CO2_transport_capita",
    unit_variable="country",
    time_variable="year",
    treatment_identifier="Sweden",
    controls_identifier=controls,
    time_optimize_ssr=range(1960, 1980),
)
synth_time = Synth()
synth_time.fit(dataprep=dp_time, optim_method="BFGS")
w_time = synth_time.weights()
yrs_time = np.arange(1960, 1991)
y_sw_t = panel_wide.loc[yrs_time, "Sweden"].values
y_sy_t = (panel_wide.loc[yrs_time, controls] @ w_time.reindex(controls).fillna(0)).values

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(yrs_time, y_sw_t, color=WARM_ORANGE, lw=2.4, label="Sweden")
ax.plot(yrs_time, y_sy_t, color=STEEL_BLUE, lw=2.2, ls="--", label="Synthetic Sweden")
ax.axvline(1980, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("In-time placebo — backdating treatment to 1980")
ax.legend(loc="lower right")
savefig("placebo_in_time")

✓ The lines stay close together right through 1990. No false-positive gap. The SCM doesn't invent gaps where no policy occurred.

## 8.2 In-space placebos — pretend each donor was treated

Re-run the entire SCM **once per country**. Each time, that country plays the role of Sweden and the other 14 are donors. We collect all 15 gap series.

If Sweden's gap is bigger than most placebo gaps, the effect is unlikely to be noise. The test statistic is the **post-/pre-treatment MSPE ratio**: deviation after treatment, normalised by pre-period fit. Bad pre-period fit penalises the unit (you can't trust a big "effect" if the model never fit you in the first place).

First, a function that runs the SCM for one country and returns its gap + MSPE stats.

In [ ]:
def run_placebo(treated_country):
    co = [c for c in countries if c != treated_country]
    try:
        dp = Dataprep(
            foo=panel,
            predictors=["GDP_per_capita", "vehicles_capita", "gas_cons_capita", "urban_pop"],
            predictors_op="mean",
            time_predictors_prior=range(1980, 1990),
            special_predictors=[
                ("CO2_transport_capita", [1989], "mean"),
                ("CO2_transport_capita", [1980], "mean"),
                ("CO2_transport_capita", [1970], "mean"),
            ],
            dependent="CO2_transport_capita",
            unit_variable="country",
            time_variable="year",
            treatment_identifier=treated_country,
            controls_identifier=co,
            time_optimize_ssr=range(1960, 1990),
        )
        sy = Synth()
        sy.fit(dataprep=dp, optim_method="BFGS")
        w = sy.weights().reindex(co).fillna(0)
        y_actual = panel_wide.loc[years, treated_country].values
        y_syn = (panel_wide.loc[years, co] @ w).values
        gap_p = y_actual - y_syn
        pre_mask = years < 1990
        post_mask = years >= 1990
        mspe_pre = (gap_p[pre_mask] ** 2).mean()
        mspe_post = (gap_p[post_mask] ** 2).mean()
        return {"country": treated_country, "gap": gap_p,
                "mspe_pre": mspe_pre, "mspe_post": mspe_post,
                "ratio": mspe_post / mspe_pre if mspe_pre > 0 else np.nan}
    except Exception as e:
        print(f"  placebo failed for {treated_country}: {e}")
        return None

Now loop over all 15 countries. This is the expensive step (~15 seconds on Colab).

In [ ]:
placebo_results = []
print("Running in-space placebos for each donor country (this takes a moment)...")
for c in countries:
    r = run_placebo(c)
    if r is not None:
        placebo_results.append(r)
print(f"\nFinished: {len(placebo_results)} placebos completed.")

Filter out placebos whose **pre-period fit was 20× worse than Sweden's**. These are units the SCM never managed to match well, so any post-period gap is meaningless. This is the standard Andersson / SCtools filter.

In [ ]:
sweden_res = next(r for r in placebo_results if r["country"] == "Sweden")
mspe_limit = 20 * sweden_res["mspe_pre"]
keep = [r for r in placebo_results
        if r["country"] == "Sweden" or r["mspe_pre"] <= mspe_limit]
print(f"Kept {len(keep)} placebos after MSPE-limit filter (limit = {mspe_limit:.4f}).")

Plot Sweden's gap (bold orange) on top of the surviving placebo gaps (grey).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
for r in keep:
    if r["country"] == "Sweden":
        continue
    ax.plot(years, r["gap"], color=LIGHT_TEXT, lw=1.0, alpha=0.55)
ax.plot(years, sweden_res["gap"], color=WARM_ORANGE, lw=2.6, label="Sweden")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.axhline(0, color=LIGHT_TEXT, lw=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Gap in per-capita CO$_2$ from transport")
ax.set_title("In-space placebos — Sweden vs donor-pool placebos")
ax.legend(loc="lower left")
savefig("placebo_in_space")

Sweden's post-1990 gap clearly stands outside the bundle of grey placebo gaps.

Quantify by the post/pre MSPE ratio. The **permutation p-value** is the fraction of units with a ratio ≥ Sweden's.

In [ ]:
ratios = pd.DataFrame(
    [{"country": r["country"], "ratio": r["ratio"]} for r in placebo_results]
).sort_values("ratio", ascending=False).reset_index(drop=True)
print("Post/Pre MSPE ratio per unit:")
print(ratios.round(3).to_string(index=False))
p_val = (ratios["ratio"] >= sweden_res["ratio"]).mean()
print(f"\nPermutation p-value for Sweden = {p_val:.4f}")
ratios.to_csv(ROOT / "tab_placebo_mspe_ratios.csv", index=False)

fig, ax = plt.subplots(figsize=(8.5, 6.0))
colors = [WARM_ORANGE if c == "Sweden" else STEEL_BLUE for c in ratios["country"]]
ax.barh(ratios["country"][::-1], ratios["ratio"][::-1], color=colors[::-1])
ax.set_xlabel("Post-/Pre-treatment MSPE ratio")
ax.set_title(f"Permutation test: MSPE ratios — p = {p_val:.3f}")
savefig("placebo_mspe_ratio")

✓ Sweden has the **highest ratio**. Permutation **p = 0.067** = 1/15, the smallest possible non-trivial value with 15 donors.

## 8.3 Leave-one-out — drop one high-weight donor at a time

We re-fit Synthetic Sweden six times, each time excluding one of the six donors with positive weight. If the headline gap is robust, none of these exclusions should erase it. (The results are pre-computed in `leave_one_out_data.dta` to keep this notebook fast.)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
excl_cols = [c for c in loo.columns if c.startswith("excl_")]
loo_long_labels = {
    "excl_unitedstates": "US excl.", "excl_belgium": "BE excl.",
    "excl_denmark": "DK excl.", "excl_greece": "GR excl.",
    "excl_newzealand": "NZ excl.", "excl_switzerland": "CH excl.",
}
for col in excl_cols:
    ax.plot(loo["Year"], loo[col], color=LIGHT_TEXT, lw=1.1, alpha=0.7,
            label=loo_long_labels.get(col, col))
ax.plot(loo["Year"], loo["synth_sweden"], color=WARM_ORANGE, lw=2.4,
        label="Synthetic Sweden")
ax.plot(loo["Year"], loo["sweden"], color=TEAL, lw=2.2, label="Sweden")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("Leave-one-out robustness — Synthetic Sweden across donor exclusions")
ax.legend(loc="lower right", ncol=2, fontsize=9)
savefig("placebo_leave_one_out")

✓ All six exclusion paths sit tightly bundled together. The estimated reduction ranges from **8.8% (without Switzerland) to 13% (without Denmark)** — all firmly negative, all bracketing the headline 11%.

**All three falsification tests pass.** The −11.3% effect is unlikely to be a method artefact.

# 9. Was GDP a confounder?

A **confounder** is a variable that affects both treatment and outcome. The most common objection to the carbon-tax story: maybe Sweden's emissions fell for economic reasons (a recession, a structural shift) and the tax just happened to coincide.

We rule this out in two steps:

1. Compare CO2 and GDP gaps side by side. If recessions caused the CO2 reduction, the CO2 gap should track the GDP gap.
2. Build a *second* Synthetic Sweden — this time using GDP as the outcome. If the carbon tax depressed growth, the GDP gap should be negative.

## 9.1 Sweden GDP and CO2 levels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(ds["year"], ds["GDP_Sweden"], color=STEEL_BLUE, lw=2.2)
axes[0].axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
axes[0].set_title("GDP per capita — Sweden")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("USD per capita (real)")

axes[1].plot(ds["year"], ds["CO2_Sweden"], color=WARM_ORANGE, lw=2.2)
axes[1].axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
axes[1].set_title("CO$_2$ per capita — Sweden")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Metric tons / capita")
plt.suptitle("Sweden: GDP and CO$_2$ levels", color=WHITE_TEXT, fontsize=13)
savefig("gdp_co2_levels")

GDP roughly doubles 1960–2005. CO2 plateaus after 1990. Levels alone don't tell us much — we need the gaps.

## 9.2 Gaps vs Synthetic Sweden, recessions shaded

The shaded bands mark Sweden's recessions: 1976–78 and 1991–93. If recessions caused the CO2 drop, the CO2 gap (right panel) should mirror the GDP gap (left panel) — dip during the recession, then rebound.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, var, color, lab in [
    (axes[0], "gap_GDP", STEEL_BLUE, "Gap GDP (USD) — Sweden − Synth(CO$_2$)"),
    (axes[1], "gap_CO2", WARM_ORANGE, "Gap CO$_2$ (t / cap) — Sweden − Synth(CO$_2$)"),
]:
    ax.axvspan(1976, 1978, color=GRID_LINE, alpha=0.55)
    ax.axvspan(1991, 1993, color=GRID_LINE, alpha=0.55)
    ax.plot(ds["year"], ds[var], color=color, lw=2.2)
    ax.axhline(0, color=LIGHT_TEXT, lw=0.6)
    ax.set_xlabel("Year")
    ax.set_ylabel(lab)
plt.suptitle("Gaps vs Synthetic Sweden(CO$_2$) — recessions shaded",
             color=WHITE_TEXT, fontsize=13)
savefig("gdp_co2_gaps")

**The GDP gap dips deeply during both recessions, then rebounds. The CO2 gap dips but never rebounds.** That asymmetry is the smoking gun: emissions did not snap back when growth did, so it was not the recession that suppressed them.

# 10. Synthetic GDP — did the carbon tax hurt growth?

Build a second synthetic control with `gdp_cap` as the outcome, using a slightly different donor pool (13 countries with comparable schooling/investment data) and economic predictors.

## 10.1 Dataprep + fit on GDP

In [ ]:
gdp = gdp_data.copy()
gdp_controls = sorted([c for c in gdp["country"].unique() if c != "Sweden"])

dp_gdp = Dataprep(
    foo=gdp,
    predictors=["investrate", "trade", "infrate"],
    predictors_op="mean",
    time_predictors_prior=range(1980, 1990),
    special_predictors=[
        ("gdp_cap", [1975], "mean"),
        ("gdp_cap", [1980], "mean"),
        ("gdp_cap", [1989], "mean"),
        ("schooling", [1975, 1980, 1985], "mean"),
    ],
    dependent="gdp_cap",
    unit_variable="country",
    time_variable="year",
    treatment_identifier="Sweden",
    controls_identifier=gdp_controls,
    time_optimize_ssr=range(1970, 1990),
)
synth_gdp = Synth()
synth_gdp.fit(dataprep=dp_gdp, optim_method="BFGS")

## 10.2 Compute the GDP series + path plot

In [ ]:
gdp_wide = gdp.pivot(index="year", columns="country", values="gdp_cap")
gdp_years = np.arange(1970, 2006)
w_gdp = synth_gdp.weights().reindex(gdp_controls).fillna(0)
gdp_actual = gdp_wide.loc[gdp_years, "Sweden"].values
gdp_synth = (gdp_wide.loc[gdp_years, gdp_controls] @ w_gdp).values

print("Synthetic-GDP donor weights (non-zero):")
print(w_gdp[w_gdp > 1e-4].sort_values(ascending=False).round(4))
print(f"\nGDP 2005 — Sweden actual: ${gdp_actual[-1]:,.0f} vs Synthetic: ${gdp_synth[-1]:,.0f}")

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(gdp_years, gdp_actual, color=WARM_ORANGE, lw=2.4, label="Sweden")
ax.plot(gdp_years, gdp_synth, color=STEEL_BLUE, lw=2.2, ls="--", label="Synthetic Sweden(GDP)")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("GDP per capita (USD, real)")
ax.set_title("Synthetic GDP — did the carbon tax hurt Swedish growth?")
ax.legend(loc="lower right")
savefig("gdp_synth")

Donor weights: Denmark 61%, Norway 20%, Finland 10%, USA 9%. Actual and synthetic GDP overlap to within **\$233 per capita** by 2005 — less than 1% of the level.

**No measurable growth penalty.** GDP is not a confounder of the CO2 result.

# 11. Tax incidence — did consumers pay?

Synthetic control told us *how much* emissions fell. The rest of the notebook zooms in on *why* consumers changed behaviour. First question: did the tax even reach the consumer? If oil companies absorb it, the price signal never lands at the pump.

We regress year-on-year **changes** in the retail price on changes in the oil price and changes in total tax:

$$\Delta p^*_t = \beta_0 + \beta_1 \Delta \Theta_t + \beta_2 \Delta T_t + \varepsilon_t$$

The **pass-through** coefficient $\beta_2$ is the share of the tax change that consumers actually pay.

In [ ]:
reg = reg_data.copy()
tax_sub = reg[["year", "p_nom", "en_tax", "CO2_tax", "oil_p", "en_CO2_tax"]].copy()
tax_sub["delta_p"] = tax_sub["p_nom"].diff()
tax_sub["delta_oil_p"] = tax_sub["oil_p"].diff()
tax_sub["delta_tax"] = tax_sub["en_CO2_tax"].diff()
m_incid = pf.feols(
    "delta_p ~ delta_oil_p + delta_tax",
    data=tax_sub.dropna(subset=["delta_p", "delta_oil_p", "delta_tax"]),
    vcov="HC1",
)
print("Tax-incidence regression (HC1):")
print(m_incid.tidy().round(4))

**Pass-through ≈ 1.15 (SE 0.15).** 95% CI roughly [0.85, 1.45]. We can't reject full pass-through. **Consumers paid the whole tax.**

# 12. OLS gasoline-consumption regressions

How strongly does Swedish demand respond to a change in:

- The **price excluding the carbon tax** ($pv_t$), or
- The **carbon tax (with VAT)** ($ct_t$)?

If consumers only care about the total price, both responses should be equal. If they differ, that tells us about behavioural channels (salience, permanence).

Andersson's log-level model:

$$\ln y_t = \beta_0 + \beta_1 pv_t + \beta_2 ct_t + \beta_3 D_t + \beta_4 X_t + \varepsilon_t$$

Log on the left, levels on the right → coefficients are **semi-elasticities**: a 1-unit rise in $x$ → a $100\beta\%$ change in $y$.

## 12.1 Four nested specifications (HC1 SEs)

We add controls one at a time. If the price and tax coefficients are stable across specifications, we are on firmer ground.

In [ ]:
ols_specs = {
    "OLS1": "log_gas_cons ~ p_real_vat + real_CO2_tax_vat + d_CO2_tax + t",
    "OLS2": "log_gas_cons ~ p_real_vat + real_CO2_tax_vat + d_CO2_tax + t + gdp_cap",
    "OLS3": "log_gas_cons ~ p_real_vat + real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop",
    "OLS4": "log_gas_cons ~ p_real_vat + real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl",
}
ols_fits = {name: pf.feols(f, data=reg, vcov="HC1") for name, f in ols_specs.items()}
for name, fit in ols_fits.items():
    print(f"\n{name} (HC1):")
    print(fit.tidy()[["Estimate", "Std. Error", "t value", "Pr(>|t|)"]].round(4))

**OLS4 (full controls):**

- Price semi-elasticity: **β = −0.060** → 1 SEK/litre higher price → **6% lower** gasoline use.
- Tax semi-elasticity: **β = −0.186** → 1 SEK/litre higher carbon tax → **18.6% lower** gasoline use.

The tax response is roughly **3× the price response**, stable across all four specs.

## 12.2 Newey–West HAC standard errors (16 lags)

HC1 corrects for heteroskedasticity but not for autocorrelation. For time-series data we want **Newey–West HAC** SEs. `pyfixest` doesn't expose them directly, so we use `statsmodels` against the same design matrix.

In [ ]:
def newey_west_table(formula, data, maxlags=16):
    rhs = formula.split("~")[1]
    ys = formula.split("~")[0].strip()
    cols = [v.strip() for v in rhs.split("+")]
    sub = data[[ys] + cols].dropna()
    X = sm.add_constant(sub[cols])
    res = sm.OLS(sub[ys], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return pd.DataFrame({
        "coef": res.params, "se_nw16": res.bse,
        "t": res.tvalues, "p": res.pvalues,
    })

nw_tables = {name: newey_west_table(f, reg) for name, f in ols_specs.items()}
print("Newey-West HAC(16) results — OLS4:")
print(nw_tables["OLS4"].round(4))

wide = pd.concat({k: v[["coef", "se_nw16"]] for k, v in nw_tables.items()}, axis=1)
wide.to_csv(ROOT / "tab_ols_newey_west.csv")

Coefficients are identical to HC1 (same OLS point estimates). The Newey-West SEs are slightly *tighter* — both elasticities still significant at p < 0.001.

# 13. Instrumental variables (2SLS) — address endogeneity

OLS is unbiased only if the regressors are uncorrelated with the error term. Otherwise the coefficient is **biased** — endogeneity.

In our setting, $pv_t$ may be endogenous: unobserved demand shocks (electric-vehicle push, EU fuel-economy regulation) could lower demand *and* shift the price. The fix is **two-stage least squares (2SLS)** with an **instrument** $z$ that satisfies:

1. **Relevance:** $z$ is correlated with $pv_t$.
2. **Exogeneity:** $z$ is *not* correlated with the error term.

Andersson proposes two:

- **Real crude oil price** — exogenous because Sweden is too small to move world oil markets.
- **Real energy tax** — exogenous because tax changes are slow and policy-driven.

## 13.1 Restrict to 1970–2011 (when all variables are available)

In [ ]:
iv_data = reg[(reg["year"] >= 1970) & (reg["year"] <= 2011)].copy()
print(f"IV sample: {iv_data.shape[0]} years")

## 13.2 Three IV specifications

`pyfixest`'s formula syntax for IV: `outcome ~ exogenous_regressors | endogenous ~ instruments`.

**IV with the crude oil price as the only instrument.**

In [ ]:
iv2 = pf.feols(
    "log_gas_cons ~ real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl "
    "| p_real_vat ~ oil_p_real",
    data=iv_data, vcov="HC1",
)
print("IV: oil_p_real instrument (HC1):")
print(iv2.tidy().round(4))

**IV with the energy tax as the only instrument.**

In [ ]:
iv1 = pf.feols(
    "log_gas_cons ~ real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl "
    "| p_real_vat ~ real_en_tax_vat",
    data=iv_data, vcov="HC1",
)
print("IV: real_en_tax_vat instrument (HC1):")
print(iv1.tidy().round(4))

**IV with both instruments together (over-identified).**

In [ ]:
iv_both = pf.feols(
    "log_gas_cons ~ real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl "
    "| p_real_vat ~ oil_p_real + real_en_tax_vat",
    data=iv_data, vcov="HC1",
)
print("IV: both instruments (HC1):")
print(iv_both.tidy().round(4))

## 13.3 First-stage diagnostic — are the instruments relevant?

A first-stage regression of the endogenous variable on the instruments tells us whether the instruments actually predict the price. We want significant coefficients on both instruments — otherwise the IV estimates are unreliable.

In [ ]:
fs_both = pf.feols(
    "p_real_vat ~ real_CO2_tax_vat + d_CO2_tax + t + gdp_cap + urban_pop + unempl + "
    "oil_p_real + real_en_tax_vat",
    data=iv_data, vcov="HC1",
)
print("First-stage regression (HC1):")
print(fs_both.tidy().round(4))

Both `oil_p_real` and `real_en_tax_vat` are strongly significant in the first stage (t ≫ 2). The instruments are relevant.

## 13.4 OLS vs IV comparison table

In [ ]:
def coef_se(fit, var):
    t = fit.tidy()
    if var in t.index:
        row = t.loc[var]
        return float(row["Estimate"]), float(row["Std. Error"])
    return None, None

iv_summary = pd.DataFrame({
    "model": ["OLS4", "IV (energy tax)", "IV (oil price)", "IV (both)"],
    "beta_p_real_vat": [
        coef_se(ols_fits["OLS4"], "p_real_vat")[0],
        coef_se(iv1, "p_real_vat")[0],
        coef_se(iv2, "p_real_vat")[0],
        coef_se(iv_both, "p_real_vat")[0],
    ],
    "beta_real_CO2_tax_vat": [
        coef_se(ols_fits["OLS4"], "real_CO2_tax_vat")[0],
        coef_se(iv1, "real_CO2_tax_vat")[0],
        coef_se(iv2, "real_CO2_tax_vat")[0],
        coef_se(iv_both, "real_CO2_tax_vat")[0],
    ],
}).round(4)
iv_summary.to_csv(ROOT / "tab_iv_comparison.csv", index=False)
print("OLS vs IV comparison:")
print(iv_summary.to_string(index=False))

**Tax coefficient = −0.186 in all four** (OLS4 + 3 IV variants). The price coefficient barely moves either. OLS and IV agreeing closely is itself evidence that endogeneity bias is small.

## 13.5 Visual comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
x_lbl = ["price semi-elasticity\n($p^v_t$)", "tax semi-elasticity\n($ct_t$)"]
x = np.arange(2)
bar_w = 0.2
models = iv_summary["model"].tolist()
colors_m = [STEEL_BLUE, WARM_ORANGE, TEAL, "#c179c8"]
for i, m in enumerate(models):
    row = iv_summary[iv_summary["model"] == m].iloc[0]
    ax.bar(x + (i - 1.5) * bar_w,
           [row["beta_p_real_vat"], row["beta_real_CO2_tax_vat"]],
           width=bar_w, color=colors_m[i], label=m)
ax.set_xticks(x)
ax.set_xticklabels(x_lbl)
ax.axhline(0, color=LIGHT_TEXT, lw=0.6)
ax.set_ylabel("Coefficient (log gasoline consumption per SEK / litre)")
ax.set_title("Price and tax semi-elasticities — OLS4 vs IV")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)
savefig("iv_vs_ols_coefs")

**Why the 3× tax-vs-price asymmetry?** Two channels:

- **Salience.** A tax change is announced, debated, visible. A market price change is a slow billboard drift.
- **Permanence.** A tax change is sticky. Market prices may revert; consumers may wait them out.

Policy implication: revenue-neutral tax swaps (raise carbon tax, cut another tax) can deliver real emission reductions even when the household tax bill is unchanged.

# 14. Disentangling carbon tax from VAT

The 1990/91 reform was a **bundle** of three things. What fraction is the carbon tax alone? Andersson simulates the demand model under three counterfactual scenarios:

| Scenario | What's switched on | What's switched off |
| --- | --- | --- |
| `CarbonTaxandVAT` (actual) | All three components | Nothing |
| `NoCarbonTaxWithVAT` | VAT + energy tax | Carbon tax |
| `NoCarbonTaxNoVAT` | Energy tax only | Carbon tax + VAT |

The vertical gap between actual and `NoCarbonTaxWithVAT` is the **carbon-tax-only** contribution.

## 14.1 Filter to 1970–2005 and peek at last 6 years

In [ ]:
dis = disent[(disent["year"] >= 1970) & (disent["year"] <= 2005)].copy()
print(dis.tail(6).round(4).to_string(index=False))

In 2005: actual (with reform) = 2.29 t/cap; without carbon tax = 2.86; without anything = 3.05. The carbon tax alone saved 0.57 t/cap that year.

## 14.2 Compute the average post-1990 carbon-tax-only reduction

In [ ]:
mask_post = dis["year"] >= 1990
dis_post = dis[mask_post]
ct_reduction_pct = (
    (dis_post["NoCarbonTaxWithVAT"] - dis_post["CarbonTaxandVAT"])
    / dis_post["NoCarbonTaxWithVAT"]
).mean() * 100
print(f"Mean post-1990 carbon-tax-attributable reduction "
      f"(rel. to no-carbon-tax-with-VAT scenario): {ct_reduction_pct:.2f}%")

**9.5%** post-1990 carbon-tax-only reduction, relative to the with-VAT-but-no-carbon-tax baseline. (Andersson reports 6.3% against the synthetic-Sweden baseline; same physical wedge, different denominator.)

## 14.3 Three-scenario plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot(dis["year"], dis["NoCarbonTaxNoVAT"], color=TEAL, lw=2.2, ls=":",
        label="No carbon tax, no VAT")
ax.plot(dis["year"], dis["NoCarbonTaxWithVAT"], color=STEEL_BLUE, lw=2.2, ls="--",
        label="No carbon tax, with VAT")
ax.plot(dis["year"], dis["CarbonTaxandVAT"], color=WARM_ORANGE, lw=2.4,
        label="Carbon tax + VAT (actual)")
ax.axvline(1990, color=LIGHT_TEXT, lw=0.8, ls=":")
ax.set_xlabel("Year")
ax.set_ylabel("Metric tons CO$_2$ / capita (transport)")
ax.set_title("Counterfactual CO$_2$ scenarios — disentangling tax components")
ax.legend(loc="upper left")
savefig("disentangling")

dis.to_csv(ROOT / "tab_disentangling.csv", index=False)

The wedge between orange and blue (carbon-tax-only) widens sharply after 2000, when the carbon-tax rate was ratcheted up. By 2005 the carbon tax alone explains **~75% of the total reform wedge**.

# 15. Headline summary — pulling it all together

In [ ]:
summary = {
    "DiD (Sweden vs Denmark) on Sweden_post":
        m_did2.tidy().loc["Sweden_post", "Estimate"],
    "DiD (Sweden vs OECD, cluster SE) on Sweden_post":
        m_did_oecd.tidy().loc["Sweden_post", "Estimate"],
    "Synthetic Sweden — 2005 gap (t CO2/cap)":
        y_sweden.loc[2005] - y_synth.loc[2005],
    "Synthetic Sweden — average post-treatment % reduction":
        avg_post_pct,
    "Permutation placebo p-value (post/pre MSPE ratio)":
        p_val,
    "OLS4 — price semi-elasticity (beta1)":
        ols_fits["OLS4"].tidy().loc["p_real_vat", "Estimate"],
    "OLS4 — tax semi-elasticity (beta2)":
        ols_fits["OLS4"].tidy().loc["real_CO2_tax_vat", "Estimate"],
    "IV (oil) — price semi-elasticity":
        iv2.tidy().loc["p_real_vat", "Estimate"],
    "IV (oil) — tax semi-elasticity":
        iv2.tidy().loc["real_CO2_tax_vat", "Estimate"],
    "Disentangling — mean carbon-tax-only reduction (%)":
        ct_reduction_pct,
}
summary_df = pd.DataFrame({"value": summary}).round(4)
print(summary_df.to_string())
summary_df.to_csv(ROOT / "tab_headline_summary.csv")
print("\n=== All done ===")

## Five numbers to remember

| Quantity | Value | What it means |
| --- | --- | --- |
| Synthetic Sweden — average gap | **−11.3%** per year (1990–2005) | The carbon tax cut transport CO2 by ~a tenth, every year, for 16 years |
| Permutation p-value | **0.067** | Only 1 in 15 placebo countries shows a gap as big as Sweden's |
| Leave-one-out range | **8.8% to 13%** | Result survives dropping any single high-weight donor |
| Tax-vs-price asymmetry | **3×** | Consumers cut consumption 3× harder per SEK of tax than per SEK of price |
| Synthetic GDP gap | **< \$233 / capita** | No detectable growth penalty |

---

## Next steps & three self-study exercises

- Read the [full blog post](https://carlos-mendez.org/post/python_sc_co2tax/) for deeper narrative.
- Try `pysyncon.AugSynth` (ridge-augmented synthetic control) — does the gap move?
- Extend the panel beyond 2005 with newer OECD data.

### Exercises

1. **Sensitivity to the donor pool.** Drop Denmark from `controls` before fitting `pysyncon.Synth`. Does the post-1990 gap shrink, stay the same, or grow?
2. **Alternative predictors.** Re-fit Synthetic Sweden with only the four economic predictors and *no* lagged CO2 levels in `special_predictors`. Does the pre-treatment fit deteriorate?
3. **Augmented synthetic control.** Replace `Synth()` with `pysyncon.AugSynth()`. Compare the headline gap and donor weights.

### Citations

- Andersson, J. J. (2019). [Carbon Taxes and CO2 Emissions: Sweden as a Case Study](https://doi.org/10.1257/pol.20170144). *AEJ: Economic Policy* 11(4), 1–30.
- Graefe, T. (2020). [RTutorCarbonTaxesAndCO2Emissions](https://github.com/TheresaGraefe/RTutorCarbonTaxesAndCO2Emissions).
- Abadie, A., Diamond, A., & Hainmueller, J. (2010, 2015). Synthetic-control method.